## Import Necessary Libraries

In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import transforms
from torch.utils.data import Dataset,DataLoader,Subset
from PIL import Image
import kagglehub

## Download Dataset

In [ ]:
path = kagglehub.dataset_download("mohitsingh1804/plantvillage")
print("path",path)

Using Colab cache for faster access to the 'plantvillage' dataset.
path /kaggle/input/plantvillage


## Setup

In [ ]:
torch.manual_seed(42)

In [ ]:
print(torch.cuda.is_available())

True


In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cuda


In [ ]:
TRAIN_PATH = os.path.join(path, "PlantVillage","train")
VAL_PATH = os.path.join(path, "PlantVillage","val")

## Transformation Pipeline

In [ ]:
transform = transforms.Compose(
    [
        transforms.Resize( (128,128) ),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.5]*3, std=[0.5]*3)
    ]
)

## Custom Dataset Class

In [ ]:
from genericpath import isfile

class MultiClassClassfication(Dataset):
  def __init__(self, root_dir,transform=None) :
    super().__init__()

    self.samples = []
    self.transform = transform

    self.classes = sorted([d for d in os.listdir(root_dir) if os.path.isdir(os.path.join(root_dir,d))])

    self.class_to_idx= {cls_name : idx  for idx,cls_name in enumerate(self.classes)}

    for class_name in self.classes:
      class_path = os.path.join(root_dir,class_name)

      for img_name in os.listdir(class_path):
        img_path = os.path.join(class_path,img_name)

        if os.path.isfile(img_path):
          label = self.class_to_idx[class_name]
          self.samples.append( (img_path,label) )


  def __len__(self):
    return len(self.samples)

  def __getitem__(self, idx) :

    img_path, label = self.samples[idx]
    image = Image.open(img_path).convert("RGB")

    if self.transform:
      image = self.transform(image)


    return image, label

## Load Data

In [ ]:
train_dataset_full = MultiClassClassfication(TRAIN_PATH, transform)
test_dataset_full = MultiClassClassfication(VAL_PATH, transform)
num_classes = len(train_dataset_full.classes)

print("Number of classes:", num_classes)
print("Full train size:", len(train_dataset_full))
print("Full test size:", len(test_dataset_full))

Number of classes: 38
Full train size: 43444
Full test size: 10861


## DataLoader

In [ ]:
pin = True if device.type == 'cuda' else False
train_loader = DataLoader(train_dataset_full, batch_size=32, shuffle=True, pin_memory=pin)
test_loader = DataLoader(test_dataset_full, batch_size=32, shuffle=False, pin_memory=pin)

## CNN

In [ ]:
class DeepCNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()


        self.block1 = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding='same'),
            nn.BatchNorm2d(32),  # Batch normalization is applied before the activation function because it normalizes the feature distribution before non-linearity, leading to more stable and efficient training.
            nn.ReLU(),

            nn.Conv2d(32, 32, kernel_size=3, padding='same'),
            nn.BatchNorm2d(32),
            nn.ReLU(),

            nn.MaxPool2d(2)
        )


        self.block2 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=3, padding='same'),
            nn.BatchNorm2d(64),
            nn.ReLU(),

            nn.Conv2d(64, 64, kernel_size=3, padding='same'),
            nn.BatchNorm2d(64),
            nn.ReLU(),

            nn.MaxPool2d(2)
        )


        self.block3 = nn.Sequential(
            nn.Conv2d(64, 128, kernel_size=3, padding='same'),
            nn.BatchNorm2d(128),
            nn.ReLU(),

            nn.Conv2d(128, 128, kernel_size=3, padding='same'),
            nn.BatchNorm2d(128),
            nn.ReLU(),

            nn.MaxPool2d(2)
        )


        self.block4 = nn.Sequential(
            nn.Conv2d(128, 256, kernel_size=3, padding='same'),
            nn.BatchNorm2d(256),
            nn.ReLU(),

            nn.Conv2d(256, 256, kernel_size=3, padding='same'),
            nn.BatchNorm2d(256),
            nn.ReLU(),

            nn.MaxPool2d(2)
        )


        # Fully Connected Layers
        self.classifier = nn.Sequential(
            nn.Flatten(),

            nn.Linear(256 * 8 * 8, 256),
            nn.ReLU(),
            nn.Dropout(0.4),

            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.4),

            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.4),

            nn.Linear(64, num_classes)
        )



    def forward(self, x):
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        x = self.block4(x)

        x = self.classifier(x)
        return x



In [ ]:
model = DeepCNN(num_classes=num_classes).to(device)

## Training Setup

In [ ]:
learning_rate = 0.001
epochs = 10
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

## Training Loop

In [ ]:
for epoch in range(epochs):
    model.train()
    total_loss = 0
    for batch_features, batch_labels in train_loader:
        batch_features = batch_features.to(device)
        batch_labels = batch_labels.to(device)
        outputs = model(batch_features)

        loss = criterion(outputs, batch_labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    avg_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch+1}/{epochs}, Loss: {avg_loss:.4f}")

Epoch 1/10, Loss: 2.5706
Epoch 2/10, Loss: 1.8116
Epoch 3/10, Loss: 1.4305
Epoch 4/10, Loss: 1.2620
Epoch 5/10, Loss: 1.1200
Epoch 6/10, Loss: 1.0322
Epoch 7/10, Loss: 0.9529
Epoch 8/10, Loss: 0.8936
Epoch 9/10, Loss: 0.8416
Epoch 10/10, Loss: 0.7933


In [ ]:
print(batch_labels.shape)
print(batch_labels[:5])

torch.Size([32])
tensor([24,  6, 17, 24, 20], device='cuda:0')


In [ ]:
print(outputs.shape)

torch.Size([32, 256, 8, 8])


## Evaluation Function

In [ ]:
def evaluate(loader):
    model.eval()
    total, correct = 0, 0
    with torch.no_grad():
        for batch_features, batch_labels in loader:
            batch_features, batch_labels = batch_features.to(device), batch_labels.to(device)
            outputs = model(batch_features)
            _, predicted = torch.max(outputs, 1)
            total += batch_labels.size(0)
            correct += (predicted == batch_labels).sum().item()
    return correct / total

### Training Evaluation

In [ ]:
train_acc = evaluate(train_loader)
print("Train Accuracy:", train_acc)

Train Accuracy: 0.8861292698646533


### Testing Evaluation

In [ ]:
test_acc = evaluate(test_loader)
print("Test Accuracy:", test_acc)

Test Accuracy: 0.8703618451339655


## Remarks


The CNN has learned meaningful patterns. Very small gap between train and test. No serious overfitting and model generalizes well.